# Advanced Modeling

Tujuan :

Setelah melalui tahap **Baseline Modeling**, **Historical Modeling**, dan **Elo Modeling**, proyek ini telah menghasilkan sekumpulan fitur yang semakin kaya dalam merepresentasikan kondisi masing-masing tim sebelum pertandingan berlangsung. Tahap berikutnya adalah mengevaluasi apakah algoritma yang lebih kompleks mampu memanfaatkan informasi tersebut secara lebih efektif dibandingkan model-model sebelumnya.

Pada notebook ini akan dilakukan eksperimen menggunakan dua algoritma *gradient boosting*, yaitu **XGBoost** dan **LightGBM**. Kedua algoritma dipilih karena dikenal memiliki performa yang baik pada berbagai permasalahan klasifikasi serta mampu menangkap hubungan non-linear antar fitur yang mungkin tidak dapat direpresentasikan secara optimal oleh model yang lebih sederhana. Agar hasil eksperimen dapat dibandingkan secara adil, seluruh konfigurasi utama akan dipertahankan sama dengan notebook pemodelan sebelumnya. Dataset yang digunakan tetap berasal dari hasil **Elo Feature Engineering**, proses pemisahan data tetap menggunakan **time-based train-test split**, dan seluruh preprocessing tetap dilakukan melalui **Pipeline** untuk menghindari *data leakage*.

Pada akhir notebook ini akan dilakukan perbandingan menyeluruh terhadap seluruh eksperimen pemodelan yang telah dilakukan, sehingga dapat ditentukan model terbaik yang akan digunakan sebagai dasar pembangunan **Match Prediction API** pada tahap berikutnya.

## 1. Load Elo Dataset

Memuat dataset hasil Elo Rating Engine yang akan digunakan sebagai dasar seluruh eksperimen Advanced Modeling. Karena kita ingin membandingkan algoritma, dataset yang dipakai harus sama dengan notebook sebelumnya.

In [9]:
import pandas as pd

from sklearn.model_selection import train_test_split

In [10]:
df = pd.read_csv("../data/processed/elo_features.csv")

In [11]:
df.head()

,date,home_team,away_team,home_score,away_score,tournament,neutral,home_last5_winrate,away_last5_winrate,home_last10_winrate,...,away_avg_goals_scored,home_avg_goals_conceded,away_avg_goals_conceded,home_goal_difference_form,away_goal_difference_form,match_result,home_elo_before,away_elo_before,home_elo_after,away_elo_after
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,False,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,D,1500.000000,1500.000000,1500.000000,1500.000000
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,False,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,H,1500.000000,1500.000000,1510.000000,1490.000000
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,False,0.000000,0.500000,0.000000,...,2.000000,2.000000,1.000000,-1.000000,1.000000,H,1490.000000,1510.000000,1500.575011,1499.424989
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,False,0.333333,0.333333,0.333333,...,1.333333,1.333333,1.666667,0.333333,-0.333333,D,1499.424989,1500.575011,1499.458089,1500.541911
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,False,0.250000,0.250000,0.250000,...,1.750000,1.750000,1.500000,-0.250000,0.250000,H,1500.541911,1499.458089,1510.510716,1489.489284


In [12]:
df.head()

,date,home_team,away_team,home_score,away_score,tournament,neutral,home_last5_winrate,away_last5_winrate,home_last10_winrate,...,away_avg_goals_scored,home_avg_goals_conceded,away_avg_goals_conceded,home_goal_difference_form,away_goal_difference_form,match_result,home_elo_before,away_elo_before,home_elo_after,away_elo_after
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,False,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,D,1500.000000,1500.000000,1500.000000,1500.000000
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,False,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,H,1500.000000,1500.000000,1510.000000,1490.000000
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,False,0.000000,0.500000,0.000000,...,2.000000,2.000000,1.000000,-1.000000,1.000000,H,1490.000000,1510.000000,1500.575011,1499.424989
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,False,0.333333,0.333333,0.333333,...,1.333333,1.333333,1.666667,0.333333,-0.333333,D,1499.424989,1500.575011,1499.458089,1500.541911
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,False,0.250000,0.250000,0.250000,...,1.750000,1.750000,1.500000,-0.250000,0.250000,H,1500.541911,1499.458089,1510.510716,1489.489284


In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 49433 entries, 0 to 49432
Data columns (total 22 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   date                       49433 non-null  str    
 1   home_team                  49433 non-null  str    
 2   away_team                  49433 non-null  str    
 3   home_score                 49433 non-null  float64
 4   away_score                 49433 non-null  float64
 5   tournament                 49433 non-null  str    
 6   neutral                    49433 non-null  bool   
 7   home_last5_winrate         49292 non-null  float64
 8   away_last5_winrate         49238 non-null  float64
 9   home_last10_winrate        49292 non-null  float64
 10  away_last10_winrate        49238 non-null  float64
 11  home_avg_goals_scored      49292 non-null  float64
 12  away_avg_goals_scored      49238 non-null  float64
 13  home_avg_goals_conceded    49292 non-null  float64
 14  a

Dataset yang digunakan merupakan hasil dari notebook Elo Rating Engine, yang telah menggabungkan tiga kelompok informasi utama:

- Baseline Features
- Historical Features
- Elo Rating

Dengan demikian, eksperimen pada notebook ini tidak lagi berfokus pada pembangunan fitur, melainkan mengevaluasi apakah algoritma Machine Learning yang lebih canggih mampu memanfaatkan informasi tersebut secara lebih efektif dibandingkan Logistic Regression dan Random Forest.

## 2. Feature & Target Separation

Tujuan : 

Memisahkan:

features (X) → informasi yang tersedia sebelum pertandingan
target (y) → hasil pertandingan yang ingin diprediksi

Kita tidak membuat feature baru di sini. Notebook ini khusus Advanced Modeling, jadi dataset hasil Elo Engine digunakan apa adanya.


### 2.1 Membentuk Target

Sama seperti notebook modeling sebelumnya, target match_result dibentuk dari skor pertandingan.

In [14]:
df["match_result"] = df.apply(
    lambda row: (
        "H" if row["home_score"] > row["away_score"]
        else "A" if row["home_score"] < row["away_score"]
        else "D"
    ),
    axis=1
)


In [15]:
df["match_result"].value_counts(normalize=True)

match_result
H    0.490098
A    0.282463
D    0.227439
Name: proportion, dtype: float64

Variabel target match_result dibentuk berdasarkan hasil akhir pertandingan dan terdiri dari tiga kelas, yaitu H (Home Win), D (Draw), dan A (Away Win). Pembentukan target dilakukan sebelum pemisahan fitur agar label yang digunakan tetap konsisten dengan eksperimen Baseline, Historical, dan Elo Modeling.

### 2.2 Menentukan Feature Columns

In [16]:
# CATEGORICAL
categorical_features = [
    "home_team",
    "away_team",
    "tournament"
]

# BOOLEAN
boolean_features = [
    "neutral"
]

# HISTORICAL
historical_features = [
    "home_last5_winrate",
    "home_last10_winrate",
    "home_avg_goals_scored",
    "home_avg_goals_conceded",
    "home_goal_difference_form",
    "away_last5_winrate",
    "away_last10_winrate",
    "away_avg_goals_scored",
    "away_avg_goals_conceded",
    "away_goal_difference_form"
]

# ELO
elo_features = [
    "home_elo_before",
    "away_elo_before"
]

# ALL
feature_columns = (
    categorical_features
    + boolean_features
    + historical_features
    + elo_features
)

### 2.3 Separation

In [17]:
X = df[feature_columns]
y = df["match_result"]

In [18]:
X.head()

,home_team,away_team,tournament,neutral,home_last5_winrate,home_last10_winrate,home_avg_goals_scored,home_avg_goals_conceded,home_goal_difference_form,away_last5_winrate,away_last10_winrate,away_avg_goals_scored,away_avg_goals_conceded,away_goal_difference_form,home_elo_before,away_elo_before
0,Scotland,England,Friendly,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1500.000000,1500.000000
1,England,Scotland,Friendly,False,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1500.000000,1500.000000
2,Scotland,England,Friendly,False,0.000000,0.000000,1.000000,2.000000,-1.000000,0.500000,0.500000,2.000000,1.000000,1.000000,1490.000000,1510.000000
3,England,Scotland,Friendly,False,0.333333,0.333333,1.666667,1.333333,0.333333,0.333333,0.333333,1.333333,1.666667,-0.333333,1499.424989,1500.575011
4,Scotland,England,Friendly,False,0.250000,0.250000,1.500000,1.750000,-0.250000,0.250000,0.250000,1.750000,1.500000,0.250000,1500.541911,1499.458089


In [19]:
y.head()

0    D
1    H
2    H
3    D
4    H
Name: match_result, dtype: str

### 2.4 Feature Leakage Check

In [20]:
forbidden_columns = [
    "home_score",
    "away_score",
    "match_result",
    "home_elo_after",
    "away_elo_after"
]

leakage_columns = [
    col for col in forbidden_columns
    if col in X.columns
]

leakage_columns

[]

Sebelum memasuki tahap pemodelan, dilakukan pemeriksaan terhadap kolom yang berpotensi menyebabkan target leakage. Kolom skor akhir, target pertandingan, serta Elo setelah pertandingan tidak digunakan sebagai fitur karena nilainya baru tersedia setelah pertandingan berlangsung. Hasil pemeriksaan menunjukkan bahwa tidak terdapat kolom yang berpotensi menyebabkan data leakage pada feature set.

### 2.5 Feature Set Final

In [21]:
print("Categorical Features:")
print(categorical_features)

print("\nBoolean Features:")
print(boolean_features)

print("\nHistorical Features:")
print(historical_features)

print("\nElo Features:")
print(elo_features)

print(f"\nTotal Features: {len(feature_columns)}")

Categorical Features:
['home_team', 'away_team', 'tournament']

Boolean Features:
['neutral']

Historical Features:
['home_last5_winrate', 'home_last10_winrate', 'home_avg_goals_scored', 'home_avg_goals_conceded', 'home_goal_difference_form', 'away_last5_winrate', 'away_last10_winrate', 'away_avg_goals_scored', 'away_avg_goals_conceded', 'away_goal_difference_form']

Elo Features:
['home_elo_before', 'away_elo_before']

Total Features: 16


## 3. Time-based Train-Test Split

Membagi dataset berdasarkan urutan waktu menjadi:

80% → Training

20% → Testing

Tanpa melakukan randomization.

### 3.1 Pastikan Dataset Terurut

In [22]:
df["date"].is_monotonic_increasing

True

### 3.2 Tentukan Split Point

In [24]:
split_index = int(len(df) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

### 3.3 Validasi Ukuran Dataset

In [25]:
print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))

print("Train ratio:", len(X_train) / len(X))
print("Test ratio :", len(X_test) / len(X))

Training samples: 39546
Testing samples : 9887
Train ratio: 0.7999919082394352
Test ratio : 0.2000080917605648


### 3.4 Validasi Rentang Waktu

In [26]:
train_dates = df["date"].iloc[:split_index]
test_dates = df["date"].iloc[split_index:]

print("Training period:")
print(train_dates.min(), "→", train_dates.max())

print("\nTesting period:")
print(test_dates.min(), "→", test_dates.max())

Training period:
1872-11-30 → 2016-03-25

Testing period:
2016-03-25 → 2026-06-18


In [27]:
train_dates.max() <= test_dates.min()

True

### 3.5 Cek Distribusi Target

In [28]:
print("=== TRAIN ===")
print(y_train.value_counts(normalize=True))

print("\n=== TEST ===")
print(y_test.value_counts(normalize=True))

=== TRAIN ===
match_result
H    0.493451
A    0.280509
D    0.226041
Name: proportion, dtype: float64

=== TEST ===
match_result
H    0.476687
A    0.290280
D    0.233033
Name: proportion, dtype: float64


Dataset dibagi menggunakan time-based split dengan proporsi 80% data awal sebagai training set dan 20% data terbaru sebagai testing set. Pendekatan ini mempertahankan urutan kronologis pertandingan dan mensimulasikan skenario prediksi dunia nyata, di mana model hanya memiliki akses terhadap pertandingan yang telah terjadi sebelum periode prediksi.

Skema pembagian ini dipertahankan konsisten dengan Baseline, Historical, dan Elo Modeling sehingga hasil Advanced Modeling dapat dibandingkan secara adil dengan eksperimen sebelumnya.